In [1]:
import os 
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

xgb_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/2_stage_model_loocv.parquet")
xgb_model_adm1_ablation = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/2_stage_model_loocv_no_N_events_5_years.parquet")
xgb_model_subnational_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/model_out_loocv_2_stg_model_adm1.parquet")
historical_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/naive_historical_model.parquet").dropna()
wind_exposed_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_naive_model.parquet").dropna()
wind_historical_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_historical_model.parquet").dropna()

In [5]:
len(xgb_model_subnational_adm1)

29427

In [2]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, matthews_corrcoef

def compute_binary_metrics(df, thresholds=[0, 15], model_name=None):
    results = []
    
    # Ensure abs_error exists
    if 'abs_error' not in df.columns:
        df['abs_error'] = (df['reported'] - df['predicted']).abs()
    
    for t in thresholds:
        # Binarize
        df['reported_bin'] = (df['reported'] > t).astype(int)
        df['predicted_bin'] = (df['predicted'] > t).astype(int)

        # Confusion matrix components
        TP = ((df['predicted_bin'] == 1) & (df['reported_bin'] == 1)).sum()
        TN = ((df['predicted_bin'] == 0) & (df['reported_bin'] == 0)).sum()
        FP = ((df['predicted_bin'] == 1) & (df['reported_bin'] == 0)).sum()
        FN = ((df['predicted_bin'] == 0) & (df['reported_bin'] == 1)).sum()

        # Binary metrics
        precision = precision_score(df['reported_bin'], df['predicted_bin'], zero_division=0)
        recall = recall_score(df['reported_bin'], df['predicted_bin'], zero_division=0)
        f1 = f1_score(df['reported_bin'], df['predicted_bin'], zero_division=0)
        accuracy = accuracy_score(df['reported_bin'], df['predicted_bin'])
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        fnr = FN / (FN + TP) if (FN + TP) > 0 else 0
        balanced_acc = (recall + specificity) / 2
        mcc = matthews_corrcoef(df['reported_bin'], df['predicted_bin']) if (TP+TN+FP+FN) > 0 else 0
        csi = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0

        # Error metrics
        mean_abs_error = df['abs_error'].mean()
        median_abs_error = df['abs_error'].median()

        results.append({
            'model': model_name,
            'threshold': t,
            'precision': precision,
            'recall': recall,
            'specificity': specificity,
            'accuracy': accuracy,
            # 'balanced_accuracy': balanced_acc,
            'fpr': fpr,
            'fnr': fnr,
            'f1': f1,
            'csi': csi,
            # 'mcc': mcc,
            'mean_abs_error': mean_abs_error,
            'median_abs_error': median_abs_error
        })
    
    return pd.DataFrame(results)


# --- Load models ---
# xgb_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/2_stage_model_loocv.parquet")
# xgb_model_adm1_ablation = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/2_stage_model_loocv_no_N_events_5_years.parquet")
# historical_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/naive_historical_model.parquet").dropna()
# wind_exposed_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_naive_model.parquet").dropna()
# wind_historical_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_historical_model.parquet").dropna()

# --- Compute for all models ---
metrics_xgb = compute_binary_metrics(xgb_model_adm1, model_name="2-stg-xgb")
metrics_xgb_ablation = compute_binary_metrics(xgb_model_adm1_ablation, model_name="2-stg-xgb-ablation")
metrics_xgb_subnational = compute_binary_metrics(xgb_model_subnational_adm1, model_name="2-stg-xgb-subnational")
metrics_hist = compute_binary_metrics(historical_model_adm1, model_name="historical")
metrics_wind_exp = compute_binary_metrics(wind_exposed_model_adm1, model_name="windspeed-exposed")
metrics_wind_hist = compute_binary_metrics(wind_historical_model_adm1, model_name="windspeed-historical")

# --- Combine all results ---
all_metrics = pd.concat([metrics_xgb, metrics_hist, metrics_wind_exp, metrics_wind_hist], ignore_index=True)
all_metrics = all_metrics.sort_values(by=['threshold', 'model']).reset_index(drop=True)

all_metrics


,model,threshold,precision,recall,specificity,accuracy,fpr,fnr,f1,csi,mean_abs_error,median_abs_error
0,2-stg-xgb,0,0.301091,0.683168,0.732897,0.725728,0.267103,0.316832,0.417971,0.264199,3.673320,0.000000
1,historical,0,0.142410,0.947834,0.033039,0.165563,0.966961,0.052166,0.247617,0.141303,4.773579,1.562578
2,windspeed-exposed,0,0.568536,0.174683,0.977542,0.861233,0.022458,0.825317,0.267252,0.154236,4.302938,0.000000
3,windspeed-historical,0,0.559307,0.154582,0.979366,0.859881,0.020634,0.845418,0.242220,0.137799,1.573811,0.000000
4,2-stg-xgb,15,0.147073,0.500000,0.937797,0.928603,0.062203,0.500000,0.227289,0.128216,3.673320,0.000000
5,historical,15,0.035408,0.053834,0.968159,0.948730,0.031841,0.946166,0.042718,0.021825,4.773579,1.562578
6,windspeed-exposed,15,0.222039,0.440457,0.966494,0.955316,0.033506,0.559543,0.295243,0.173188,4.302938,0.000000
7,windspeed-historical,15,0.064103,0.008157,0.997414,0.976393,0.002586,0.991843,0.014472,0.007289,1.573811,0.000000


In [4]:
metrics_xgb_ablation

,model,threshold,precision,recall,specificity,accuracy,fpr,fnr,f1,csi,mean_abs_error,median_abs_error
0,2-stg-xgb-ablation,0,0.293032,0.699906,0.715585,0.713324,0.284415,0.300094,0.413107,0.260324,3.784119,0.0
1,2-stg-xgb-ablation,15,0.202513,0.443366,0.962546,0.951643,0.037454,0.556634,0.278031,0.161461,3.784119,0.0


In [3]:
metrics_xgb_subnational

,model,threshold,precision,recall,specificity,accuracy,fpr,fnr,f1,csi,mean_abs_error,median_abs_error
0,2-stg-xgb-subnational,0,0.672707,0.140028,0.988525,0.866211,0.011475,0.859972,0.231805,0.131097,1.537326,0.0
1,2-stg-xgb-subnational,15,0.345679,0.317152,0.987122,0.973052,0.012878,0.682848,0.330802,0.198180,1.537326,0.0


In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, cohen_kappa_score

def categorize_values(series):
    """
    Categorize numeric values into 3 ordinal categories:
    0: (-inf, 0]
    1: (0, 15]
    2: (15, inf)
    """
    return pd.cut(series, bins=[-np.inf, 0, 15, np.inf], labels=[0,1,2]).astype(int)

def compute_distance_metrics_categorized_mape(models_dict):
    """
    Compute distance/error-based metrics for multiple models with categorized QWK and MAPE.
    """
    results = []

    for name, df in models_dict.items():
        # Ensure abs_error exists
        if 'abs_error' not in df.columns:
            df['abs_error'] = (df['reported'] - df['predicted']).abs()

        mae = df['abs_error'].mean()
        medae = df['abs_error'].median()
        rmse = np.sqrt(mean_squared_error(df['reported'], df['predicted']))
        
        # MAPE: only where reported > 0
        mask = df['reported'] != 0
        if mask.sum() > 0:
            mape = (df.loc[mask, 'abs_error'] / df.loc[mask, 'reported']).mean() * 100
        else:
            mape = np.nan
        
        # Categorize for QWK
        reported_cat = categorize_values(df['reported'])
        predicted_cat = categorize_values(df['predicted'])
        qwk = cohen_kappa_score(reported_cat, predicted_cat, weights='quadratic')
        
        results.append({
            'model': name,
            'MAE': mae,
            'MedAE': medae,
            'RMSE': rmse,
            # 'MAPE (%)': mape,
            'QWK': qwk
        })
    
    return pd.DataFrame(results)


# --- Example usage ---
models_dict = {
    "2-stg-XGBoost": xgb_model_adm1,
    "2-stg-XGBoost-ablation": xgb_model_adm1_ablation,
    "2-stg-XGBoost-subnational": xgb_model_subnational_adm1,
    "historical": historical_model_adm1,
    "windspeed-exposed": wind_exposed_model_adm1,
    "windspeed-historical": wind_historical_model_adm1
}

distance_metrics_df = compute_distance_metrics_categorized_mape(models_dict)
distance_metrics_df


,model,MAE,MedAE,RMSE,QWK
0,2-stg-XGBoost,3.673320,0.000000,9.523935,0.290940
1,2-stg-XGBoost-ablation,3.784119,0.000000,9.418890,0.308511
2,2-stg-XGBoost-subnational,1.537326,0.000000,8.413186,0.277654
3,historical,4.773579,1.562578,13.930599,-0.004005
4,windspeed-exposed,4.302938,0.000000,19.181691,0.306679
5,windspeed-historical,1.573811,0.000000,9.988413,0.187550


Performance in the Philippines

In [11]:
xgb_model_adm1["GID_0"] = xgb_model_adm1["DisNo."].apply(lambda x: x[-3:])

In [12]:
phl_events = xgb_model_adm1[xgb_model_adm1.GID_0=="PHL"].copy()
compute_binary_metrics(phl_events, model_name="2-stg-xgb")

,model,threshold,precision,recall,specificity,accuracy,fpr,fnr,f1,csi,mean_abs_error,median_abs_error
0,2-stg-xgb,0,0.333571,0.522613,0.816613,0.772689,0.183387,0.477387,0.407222,0.255668,3.595946,0.0
1,2-stg-xgb,15,0.154717,0.518987,0.923235,0.912579,0.076765,0.481013,0.238372,0.135314,3.595946,0.0
